# Topic 5. Recursive functions and data types

The goals of this topic are to understand:

* How recursive types (lists, trees, etc.) are defined algebraically
* How functions over recursive types are defined recursivelly
* The two major types of recursive functions: general and tail-recursive

## Recursive types

### The `List` type

Lists are data structures which represent sequences of values of the same type, of finite length. They can be defined recursively in an informal way as follows: 
- A list is the empty sequence
- A list is a non-empty sequence made of a value and another list, which represent the head and tail of the list, respectively

Thus, the type `IntList`, which represents lists of integers, must satisfy the following algebraic equation:

`IntList = 1 + Int * IntList`

i.e., a list of integers is the empty sequence (represented by the singleton type `1`), or an integer (the head) and a list (its tail).



The implementation in Scala is similar to the following one (we also give the generic version `List[A]`, rather than the implementation of `IntList`):

In [ ]:
object StdDefinition {
    // type Either[A, B] = A + B 
    sealed abstract class Either[A, B]
    case class Left[A, B](a: A) extends Either[A, B]
    case class Right[A, B](b: B) extends Either[A, B]
    
    // type IntList = 1 + Int * IntList
    sealed abstract class IntList
    case class EmptyInt() extends IntList
    case class NonEmptyInt(head: Int, tail: IntList) extends IntList
    
    sealed abstract class StringList
    case class EmptyStr() extends StringList
    case class NonEmptyStr(head: String, tail: StringList) extends StringList
    
    sealed abstract class List[A]
    case class Empty[A]() extends List[A]
    case class NonEmpty[A](head: A, tail: List[A]) extends List[A]
    
    // [1 2 3 4] : IntList
    val l: IntList = 
        NonEmptyInt(1, NonEmptyInt(2, NonEmptyInt(3, NonEmptyInt(4, EmptyInt()))))
    
    val l2: List[Int] = 
        NonEmpty[Int](1, 
            NonEmpty[Int](2, 
                NonEmpty[Int](3, 
                    NonEmpty[Int](4, 
                        Empty[Int]()))))
    
    val l3: List[Int] = 
        NonEmpty(1, 
            NonEmpty(2, 
                NonEmpty(3, 
                    NonEmpty(4, 
                        Empty()))))
    
    val l4: List[Int] = Empty()
}

However the actual implementation of [immutable lists](https://github.com/scala/scala/blob/v2.13.1/src/library/scala/collection/immutable/List.scala#L79) in the standard library of Scala defines the empty list as an object, rather than a class. This forces us to declare the list covariantly in its generic parameter `A`, which is somewhat inconvenient at times.  The standard definition looks like as follows:

In [ ]:
object ActualStdDefinition{
    /*
    sealed abstract class List[A]
    case class Nil[A]() extends List[A]
    case class ::[A](head: A, tail: List[A]) extends List[A]

    val n: List[Int] = Nil()
    */
    
    sealed abstract class List[+A]
    case object Nil extends List[Nothing]
    case class ::[A](head: A, tail: List[A]) extends List[A]

    val n: List[Int] = Nil
}
/*
    val l3: List[Int] = 
        NonEmpty(1, 
            NonEmpty(2, 
                NonEmpty(3, 
                    NonEmpty(4, 
                        Empty()))))
  */  
val l0: List[Int] = List(1, 2, 3, 4)
val l1: List[Int] = 1 :: (2 :: (3 :: (4 :: Nil)))
val l2: List[Int] = ::(1, ::(2, ::(3, ::(4, Nil))))
val l3: List[Int] = 1 :: 2 :: 3 :: 4 :: Nil
val l4: List[Int] = Nil


### Some syntactic sugar

Note that we can write standard lists with a more compact syntax: 

In [ ]:
// Less beautifully 


// More idiomatically


And can also pattern match lists, similarly:

In [ ]:
def isEmpty1(l: List[Int]): Boolean = 
    l match {
        case Nil => 
            ??? : Boolean 
        case ::(head, tail) => 
            ??? : Boolean
    }

def isEmpty2(l: List[Int]): Boolean = 
    l match {
        case Nil => 
            true : Boolean 
        case head :: tail => 
            false : Boolean
    }

def isEmpty(l: List[Int]): Boolean = 
    l match {
        case Nil => 
            true : Boolean 
        case _ :: _ => 
            false : Boolean
    }

isEmpty(Nil)
isEmpty(List(1))
isEmpty(1 :: Nil)
isEmpty(List(1,2,3))
isEmpty(1::2::3::Nil)
// Less beautifully

// more idiomatically


// or


##  Recursive functions

Since lists are defined recursively, functions over lists will be commonly recursive as well. For instance, let's implement a recursive function that computes the length of a list. But before, let's implement the function imperatively for the sake of comparison:

In [ ]:
(1 :: 2 :: Nil).isEmpty
Nil.isEmpty

In [ ]:
// Using mutable variables

def lengthI(l: List[String]): Int = {
    var out: Int = 0
    var aux: List[String] = l
    while (!aux.isEmpty){
        out += 1
        aux = aux.tail
    }
    return out
}

In [ ]:
lengthI(List("1","2", "3", "4"))
lengthI(Nil)

The recursive function is implemented as follows: 

In [ ]:
// Using recursive functions

def lengthR(l: List[String]): Int = 
    ??? : Int

In [ ]:
// Using recursive functions

def lengthR(l: List[String]): Int = 
    l match {
        case Nil => 
            ??? : Int
        case head :: tail => 
            ??? : Int
    }

In [ ]:
// Using recursive functions

def lengthR(l: List[String]): Int = 
    l match {
        case Nil => 
            0 : Int
        case head :: tail => 
            val tailSol: Int = ??? 
            ??? : Int
    }

In [ ]:
// Using recursive functions

def lengthR(l: List[String]): Int = 
    l match {
        case Nil => 
            0 : Int
        case head :: tail => 
            val tailSol: Int = lengthR(tail)
            1+tailSol : Int
    }

In [ ]:
// Using recursive functions

def lengthR(l: List[String]): Int = 
    l match {
        case Nil => 
            0
        case _ :: tail => 
            1+lengthR(tail)
    }

In [6]:
// Using recursive functions

def lengthR[A](l: List[A]): Int = 
    l match {
        case Nil => 
            0
        case _ :: tail => 
            1+lengthR(tail)
    }

defined function lengthR

In [ ]:
lengthR(List("1", "hola", "adios", "blba"))
lengthR(List(1,2,3))
lengthR(Nil)

Some comments: 
- The recursive function is implemented in a _type-driven development_ style: we proceed, step-by-step, analysing the types of input data that we have available so far, and the types of output that we have to generate. This leads to a divide-and-conquer problem solving strategy and hugely facilitates the implementation.
- The recursive function is less efficient, since the stack will blow up with lists of enough lenght.

### Tail-recursive functions

The implementation using tail-recursion solves the problems with the stack. It commonly makes use of auxiliary functions:

In [1]:
// Using tail-recursive functions

def lengthT[A](l: List[A]): Int = {
    
    def lengthAux(out: Int, aux: List[A]): Int = 
        if (aux.isEmpty) out // terminar
        else lengthAux(out+1, aux.tail) // actualizar variables y realizar una nueva pasada
    
    lengthAux(0, l)
    /*
    var out: Int = 0
    var aux: List[String] = l
    while (!aux.isEmpty){
        out += 1
        aux = aux.tail
    }
    return out*/
}


defined function lengthT

In [7]:
lengthT(List(1,2,3,4,5,6,7,8))
lengthT(L)
lengthR(L)

: 

We can check the stack-safety problems of non-tail recursive functions, by calculating the length of a very big list. We will use the following function, which creates a constant list of given length.

In [ ]:
// First, imperatively



In [ ]:
// Next, tail-recursively



We can also use the function [`fill`](https://www.scala-lang.org/api/2.13.3/scala/collection/immutable/List$.html#fill[A](n:Int)(elem:=%3EA):CC[A]) of the Scala standard library.

In [ ]:
List.fill(5)("")
lengthR(List.fill(100)(6))
lengthR(List.fill(0)(1))
lengthI(List.fill(1000)("0"))

In [4]:
val L: List[String] = List.fill(10000000)("")

L: List[String] = List(
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
  "",
...

In [ ]:
lengthI(L)

In [ ]:
lengthR(L)

In [ ]:
// Using recursive functions

def lengthR[A](l: List[A]): Int = 
    l match {
        case Nil => 
            0
        case _ :: tail => 
            1+lengthR(tail)
    }

In [ ]:
// Using recursive functions

def lengthR[A](l: List[A]): Int = 
    l match {
        case Nil => 
            0
        case _ :: tail => 
            1+lengthR(tail)
    }

Now, let's calculate the length of a list long enough, using each of the three implementations:

In [ ]:
// Imperatively


In [ ]:
// Tail-recursive


In [ ]:
// Plain recursive


### Unit testing with `scalatest`

In [ ]:
import $ivy.`org.scalatest::scalatest:3.0.8`
import org.scalatest._

From now on, we will also make extensive use of unit testing for the different functions that we implement. And we will use the [`scalatest`](http://www.scalatest.org/) library for that purpose. In particular, for each function we will implement a test catalogue that test it against different test cases. For instance, this is a possible test class for the `length` function:

In [ ]:
/*
assert(length(List()) == 0)
assert(length(List(1)) == 1)
assert(length(List(1,2,3,4)) == 4)
*/

The method `shouldBe` is a _matcher_. The scalatest library offers an extensive catalogue of [them](http://www.scalatest.org/user_guide/using_matchers). Similarly, scalatest also support many different [testing styles](http://www.scalatest.org/user_guide/selecting_a_style). The chosen one here was `FlatSpec`. In order to execute the test catalogue we can simply use the scalatest method `run`:

### Example: adding numbers

Let's implement a function that sums all the numbers of a list.

In [ ]:
class TestSum extends FlatSpec with Matchers{
    "sum" should "work" in {
    }
}

In [ ]:
// Recursively



In [ ]:
// With tail-recursion



### Example: multiplying list elements

Let's multiply the elements of a list. If the list is empty we return the identity element for integers.

In [ ]:
object TestProduct extends FlatSpec with Matchers{
    "product" should "work" in {
    }
}

This is the common recursive implementation:

It works as expected: 

In [ ]:
run(TestProduct)

But we can optimize the function a little bit. Note that if the number 0 belongs to the list, then the result is 0, no matter how many elements the list has. So, once we find the element 0 it's a waste of resources to make the recursive call. Let's take this into account.

In [ ]:
/*def product(list: List[Int]): Int =
    list match {
        case Nil => 1
        case head :: tail => head * product(tail)
    }
    */

In [ ]:
run(TestProduct)

A similar optimization can be made for the tail-recursive implementation.

### Example: membership

Let's implement a function that given a list and an element, returns whether the element belongs to that list.

In [ ]:
object TestMember extends FlatSpec with Matchers{
    "member" should "work" in {
    }
}

In [ ]:
run(TestMember)

We can also pattern match against a specific value as follows:

### Example: last element

Let's implement a function that returns the last element of a given list. Note that an empty list does not have elements, and, hence, does not have a last element.

In [ ]:
object TestLast extends FlatSpec with Matchers{
    "last" should "work" in {
    }
}

In [ ]:
run(TestLast)

### Example: insert last

Now, a function that allows us to insert an element at the end of the list. 

In [ ]:
object TestInsertLast extends FlatSpec with Matchers{
    "insertLast" should "work" in {
    }
}

In [ ]:
run(TestInsertLast)

### Example: reverse lists

Implement a function which receives a list and returns its reverse.

In [ ]:
object TestReverse extends FlatSpec with Matchers{
    "reverse" should "work" in {
    }
}

In [ ]:
// Recursively: Really inefficient 


In [ ]:
run(TestReverse)

In [ ]:
// Tail-recursive, efficiently


In [ ]:
run(TestReverse)

### Example: concatenate lists

Let's implement this function step-by-step, following the types. We start from the signature of the desired function:

In [ ]:
object TestConcatenate extends FlatSpec with Matchers{
    "concatenate" should "work" in {
    }
}

In [ ]:
// Recursive

In [ ]:
run(TestConcatenate)

In [ ]:
// Tail-recursive

In [ ]:
run(TestConcatenate)